# FinAI Agentic RAG — Q&A with Data_Analyst & Market_Analyst

**Goal**: Use `fin_ai.agents` modules with the **local vector DB** infrastructure 
from `financial_analyst_dashboard.py` to run RAG-powered Q&A.

**Agents used**:
- `Data_Analyst` — Analyses data from RAG-retrieved context
- `Market_Analyst` — Collects financial info & aggregates client requirements

**Local RAG**: FAISS vector stores managed by `fin_ai.core.dashboard_engine`

## Architecture

```
┌─────────────────────────────────────────────────────┐
│  SingleAssistantRAG                                 │
│  ┌──────────────┐   ┌────────────────────────────┐  │
│  │ Data_Analyst │──▶│  Local RAG Query Tool       │  │
│  │  (Agent)     │   │  (wraps dashboard_engine)   │  │
│  └──────────────┘   │                            │  │
│                     │  ┌──────────────────────┐  │  │
│  ┌──────────────┐   │  │ FAISS Vector Stores  │  │  │
│  │Market_Analyst│──▶│  │ (NVDA_report, etc.)  │  │  │
│  │  (Agent)     │   │  └──────────────────────┘  │  │
│  └──────────────┘   └────────────────────────────┘  │
└─────────────────────────────────────────────────────┘
```

**Key files used**:
- `src/fin_ai/agents/workflow.py` — `SingleAssistant`, `SingleAssistantRAG`
- `src/fin_ai/agents/scheduler.py` — `AgentPipeline`, `AgentScheduler`
- `src/fin_ai/core/dashboard_engine.py` — `load_vector_stores_for_query`, `answer_question`
- `dashboard/financial_analyst_dashboard.py` — Streamlit UI (reference implementation)

In [ ]:
# ============================================================================
# 1. Imports — Agents + Dashboard RAG
# ============================================================================

import os
import sys
import json
from pathlib import Path

# Ensure the project root is on sys.path
_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

# --- autogen agent framework ---
import autogen
from autogen import register_function

from fin_ai.agents.workflow import SingleAssistant, SingleAssistantRAG
from fin_ai.agents.scheduler import (
    AgentTask,
    AgentPipeline,
    AgentScheduler,
    run_agent_direct,
)

# --- Dashboard local RAG infrastructure ---
from dashboard import (
    VECTOR_DB_DIR,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_EMBEDDINGS_PROVIDER,
    OLLAMA_BASE_URL,
    DEFAULT_CHAT_MODEL,
)
from dashboard.utils import get_embeddings

from fin_ai.core.fin_ai_engine import (
    get_source_vector_stores,
    get_vector_db_names,
    load_vector_stores_for_query,
    build_query_source_configs,
    answer_question,
)

print("✅ All imports successful.")
print(f"   Vector DB path: {VECTOR_DB_DIR}")

## LLM Configuration

Use either Ollama (local) or OpenAI-compatible API.  For OpenAI, set up `OAI_CONFIG_LIST`.

In [ ]:
# ============================================================================
# 2. LLM Configuration
# ============================================================================

# Option A: OpenAI-compatible (requires OAI_CONFIG_LIST in project root)
openai_available = Path("../OAI_CONFIG_LIST").exists()

if openai_available:
    llm_config = {
        "config_list": autogen.config_list_from_json(
            "../OAI_CONFIG_LIST",
            filter_dict={"model": ["gpt-4o", "gpt-4-0125-preview"]},
        ),
        "timeout": 120,
        "temperature": 0,
    }
    chat_provider = "openai"
else:
    # Option B: Fallback to Ollama (local)
    llm_config = {
        "config_list": [
            {
                "model": DEFAULT_CHAT_MODEL,
                "base_url": OLLAMA_BASE_URL,
                "api_key": "ollama",
            }
        ],
        "timeout": 120,
        "temperature": 0,
    }
    chat_provider = "ollama"

print(f"🔧 LLM provider: {chat_provider}")
print(f"   Model: {llm_config['config_list'][0].get('model', 'auto')}")

## Discover & Load Local Vector DBs

The dashboard manages FAISS vector stores in `vector_db/`.  Each store
has an `.faiss` index and an `.embedding.json` metadata file.  We discover
them, load the ones with source documents, and build retriever configs.

In [ ]:
# ============================================================================
# 3. Discover and load local vector DBs
# ============================================================================

# Step 3a — Discover all vector stores
source_vector_stores = get_source_vector_stores()

# List stores that have a matching source document
available_dbs = get_vector_db_names(source_vector_stores)

print(f"📂 Vector DB directory: {VECTOR_DB_DIR}")
print(f"   Stores discovered: {len(source_vector_stores)}")
print(f"   Stores with source doc: {len(available_dbs)}")
print()
for name in available_dbs:
    path = source_vector_stores[name]
    size_mb = path.stat().st_size / (1024 * 1024) if path.is_file() else 0
    print(f"   • {name}  ({size_mb:.1f} MB)")

# If no stores found, we can still use the agent with empty RAG
if not available_dbs:
    print("\n⚠️  No vector DBs with source documents found.")
    print("   Upload documents via the Streamlit dashboard first:")
    print("   streamlit run dashboard/financial_analyst_dashboard.py")

User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------
Data_Analyst (to User_Proxy):

***** Suggested tool call (call_89dEBuptUkQCyH9KKfj3MTa6): retrieve_content *****
Arguments: 
{"message":"Microsoft's 2023 income statement analysis","n_results":5}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Trying to create collection.


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


doc_ids:  [['doc_0']]
Adding content of doc doc_0 to context.
User_Proxy (to Data_Analyst):

User_Proxy (to Data_Analyst):

***** Response from calling tool (call_89dEBuptUkQCyH9KKfj3MTa6) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to your requirements, or ask for more contexts.

Your current query is: Microsoft's 2023 income statement analysis

Retrieved context is: Equity Research Report: Microsoft Corporation
Income Summarization
The company experienced a 7% Year-over-Year increase in revenue, driven by
significant contributions from its Intelligent Cloud and Productivity and Business
Processes segments, indicating a strong demand for cloud-based solutions and
productivity software. Despite the revenue growth, the Cost of Goods Sold (COGS)
increased by 5%, suggesting a need for closer cost control measures to improve c

### Load Vector Stores for Querying

Select one or more stores to load.  The embedding model must match the one
used during indexing (stored in ``.embedding.json``).

In [ ]:
# ============================================================================
# 4. Load selected vector stores and build retriever configs
# ============================================================================

# Choose which vector DBs to query (pick from available_dbs above)
selected_dbs = available_dbs[:]  # Use all available — adjust as needed
if not selected_dbs:
    selected_dbs = []

print(f"🔍 Loading {len(selected_dbs)} vector store(s)...")

# Create embeddings matching the stored model
embeddings = get_embeddings(
    model_name=DEFAULT_EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
    provider=DEFAULT_EMBEDDINGS_PROVIDER,
)

# Load FAISS stores
loaded_stores = load_vector_stores_for_query(
    selected_vector_db_names=selected_dbs,
    source_vector_stores=source_vector_stores,
    embeddings=embeddings,
)

# Build SourceRetrieverConfig list for the query engine
source_configs = build_query_source_configs(loaded_stores, group_by="vector_db")

print(f"✅ Loaded {len(loaded_stores)} stores, {len(source_configs)} retriever configs.")
for cfg in source_configs:
    print(f"   • {cfg.name}  ({cfg.description[:60]}...)" if cfg.description else f"   • {cfg.name}")

User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Data_Analyst (to User_Proxy):

***** Suggested tool call (call_5p8Z5RJeBwMQa8EWxgWdLDJz): retrieve_content *****
Arguments: 
{"message":"Microsoft's 2023 income statement details"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION retrieve_content...
Trying to create collection.
doc_ids:  [['doc_17', 'doc_18', 'doc_42']]
Adding content of doc doc_17 to context.
Adding content of doc doc_18 to context.
Adding content of doc doc_42 to context.
User_Proxy (to Data_Analyst):

User_Proxy (to Data_Analyst):

***** Response from calling tool (call_5p8Z5RJeBwMQa8EWxgWdLDJz) *****
Below is the context retrieved from the required file based on your query.
If you can't answer the question with or without the current context, you should try using a more refined search query according to your requirements, or ask for more contexts.

Your current query is: Mi

---

## SingleAssistantRAG — Data_Analyst with Local RAG

We now create a **Data_Analyst** agent that can query the local vector stores.
The agent uses autogen's `SingleAssistantRAG` workflow.  We wrap the
dashboard's `answer_question` as a callable tool and register it with the agent.

### How it works

1. Agent receives a user query (e.g. *"What are NVIDIA's key risks?"*)
2. Agent calls the `query_local_rag` tool with a refined search query
3. The tool runs `answer_question` against the loaded FAISS stores
4. Retrieved context is returned to the agent
5. Agent synthesizes the answer and responds

In [ ]:
# ============================================================================
# 5. Define the local RAG query tool
# ============================================================================

from typing import Annotated

def query_local_rag(
    query: Annotated[
        str,
        "Refined search query for the local vector stores. "
        "Be specific — e.g. 'NVIDIA revenue growth Q4 2024', "
        "'risk factors for NVDA', 'Shaw Communications market position'."
    ],
) -> str:
    """Query the local FAISS vector stores for financial context.

    This tool searches indexed documents (annual reports, research reports,
    earnings transcripts, news articles) and returns the most relevant
    excerpts.  Use it whenever you need factual context about a company.
    """
    if not source_configs:
        return "[Local RAG] No vector stores loaded. Upload documents first."

    result = answer_question(
        question=query,
        source_configs=source_configs,
        provider=chat_provider,
        system_prompt=(
            "You are a precise financial research assistant. "
            "Answer based ONLY on the retrieved context below. "
            "If the context doesn't contain relevant information, say so clearly."
        ),
        temperature=0.1,
        retrieval_mode="ensemble",
        auto_truncate_prompt=True,
    )

    response = result.get("response", "")
    elapsed = result.get("elapsed", 0)
    print(f"[Local RAG] Query answered in {elapsed:.1f}s")
    return response


print("✅ query_local_rag tool defined.")
print(f"   Targets {len(source_configs)} retriever config(s) across {len(loaded_stores)} store(s).")

### Create the Data_Analyst Agent

Register the ``query_local_rag`` tool so the agent can fetch context
from the local vector stores before answering.

In [ ]:
# ============================================================================
# 6. Create SingleAssistantRAG Data_Analyst
# ============================================================================

# We use a minimal retrieve_config — the actual RAG logic is in the registered tool
data_analyst = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=8,
    retrieve_config={
        "task": "qa",
        "vector_db": None,
        "docs_path": [],            # No doc path needed — tool handles retrieval
        "chunk_token_size": 1000,
        "get_or_create": False,
        "collection_name": "local_rag_qa",
        "must_break_at_empty_line": False,
        "customized_prompt": (
            "Below is context retrieved from the local vector stores.\n"
            "Your query: {input_question}\n"
            "Retrieved context: {input_context}\n"
            "Answer using only the context. If insufficient, ask for a refined query."
        ),
    },
    rag_description=(
        "Query the local FAISS vector stores for financial context. "
        "Use this tool to search indexed documents (annual reports, research "
        "reports, transcripts, news) before answering any factual question."
    ),
)

# Override the default RAG function with our local RAG tool
register_function(
    query_local_rag,
    caller=data_analyst.assistant,
    executor=data_analyst.user_proxy,
    name="query_local_rag",
    description=query_local_rag.__doc__,
)

print("✅ Data_Analyst agent ready with local RAG tool.")

### Run Q&A — Ask about any indexed company

Just provide a natural-language question.  The agent will use the
``query_local_rag`` tool to fetch relevant context before answering.

In [ ]:
# ============================================================================
# 7. Q&A with Data_Analyst
# ============================================================================

# Example: query about NVIDIA (use whatever is in your local vector DBs)
question = (
    "What are NVIDIA's key growth drivers and competitive risks? "
    "Provide a structured analysis with specific data points."
)

print(f"❓ Question: {question}\n")
data_analyst.chat(question)

### Ask Another Question

You can continue querying — each ``.chat()`` call resets the agent.

In [ ]:
data_analyst.chat(
    "Summarize the main risks mentioned in NVIDIA's reports."
)

---

## Market_Analyst — RAG + YFinance Tools

The ``Market_Analyst`` agent is designed to collect financial information
and aggregate it.  We give it both the local RAG tool **and** live
YFinance data tools via ``fin_ai.agents.tools``.

In [ ]:
# ============================================================================
# 8. Create Market_Analyst with RAG + YFinance tools
# ============================================================================

import fin_ai.core.tools as yf_tools

market_analyst = SingleAssistantRAG(
    "Market_Analyst",
    llm_config,
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    retrieve_config={
        "task": "qa",
        "vector_db": None,
        "docs_path": [],
        "chunk_token_size": 1000,
        "get_or_create": False,
        "collection_name": "market_rag",
        "must_break_at_empty_line": False,
        "customized_prompt": (
            "Context from local stores:\n{input_context}\n\n"
            "Query: {input_question}"
        ),
    },
    rag_description="Query local vector stores for market and company data.",
)

# Register local RAG tool
register_function(
    query_local_rag,
    caller=market_analyst.assistant,
    executor=market_analyst.user_proxy,
    name="query_local_rag",
    description=query_local_rag.__doc__,
)

# Register YFinance tools for live market data
register_function(
    yf_tools.get_stock_info,
    caller=market_analyst.assistant,
    executor=market_analyst.user_proxy,
    name="get_stock_info",
    description="Get latest stock information (price, market cap, PE ratio, etc.) for a ticker symbol.",
)
register_function(
    yf_tools.get_stock_data,
    caller=market_analyst.assistant,
    executor=market_analyst.user_proxy,
    name="get_stock_data",
    description="Retrieve historical stock price data for a ticker within a date range (YYYY-MM-DD).",
)
register_function(
    yf_tools.get_company_info,
    caller=market_analyst.assistant,
    executor=market_analyst.user_proxy,
    name="get_company_info",
    description="Get company information (name, industry, sector, country, website).",
)

print("✅ Market_Analyst agent ready with local RAG + YFinance tools.")

### Market Analysis — RAG context + live data

The agent can pull context from local docs AND fetch live stock data.

In [ ]:
# ============================================================================
# 9. Market_Analyst Q&A
# ============================================================================

market_analyst.chat(
    "Get the latest stock info for NVDA, "
    "then retrieve any relevant context from the local documents about NVIDIA. "
    "Provide a summary combining both live data and document context."
)

---

## Multi-Agent Pipeline — Data_Analyst + Market_Analyst

For complex workflows, use ``AgentPipeline``.  This pipeline:

1. **Data_Analyst** — Queries local RAG for deep document analysis
2. **Market_Analyst** — Fetches live data + aggregates market context
3. **Synthesis** — Combines both perspectives into a report

In [ ]:
# ============================================================================
# 10. Build a multi-agent pipeline
# ============================================================================

# Shared RAG config for pipeline agents
shared_rag_config = {
    "task": "qa",
    "vector_db": None,
    "docs_path": [],
    "chunk_token_size": 1000,
    "get_or_create": False,
    "collection_name": "pipeline_rag",
    "must_break_at_empty_line": False,
}

pipeline = AgentPipeline(
    name="NVIDIA_Analysis_Pipeline",
    llm_config=llm_config,
)

# Task 1: Data_Analyst — deep document analysis via local RAG
pipeline.add_task(AgentTask(
    name="deep_analysis",
    prompt=(
        "Query the local documents for NVIDIA. "
        "Analyze: key business segments, competitive position, "
        "risk factors, and management outlook. "
        "Provide a structured summary based on document evidence."
    ),
    agent_config="Data_Analyst",
    metadata={"retrieve_config": shared_rag_config},
))

# Task 2: Market_Analyst — live data + RAG
pipeline.add_task(AgentTask(
    name="market_check",
    prompt=(
        "Get the latest stock info for NVDA. "
        "Query local documents for market context. "
        "Summarize the current market position: price, valuation, "
        "analyst sentiment, and recent news."
    ),
    agent_config="Market_Analyst",
    metadata={"retrieve_config": shared_rag_config},
))

# Task 3: Synthesize (runs after tasks 1 & 2)
pipeline.add_task(AgentTask(
    name="synthesize",
    prompt=(
        "Combine the deep analysis and market check results. "
        "Produce a concise investment brief covering: "
        "1) Business fundamentals 2) Market position 3) Key risks "
        "4) Overall assessment (Bullish / Neutral / Bearish)."
    ),
    agent_config="Financial_Analyst",
    depends_on=["deep_analysis", "market_check"],
))

print("✅ Pipeline ready with 3 tasks:")
for t in pipeline._tasks:
    deps = f" → depends on: {', '.join(t.depends_on)}" if t.depends_on else ""
    print(f"   {t.name} [{t.agent_config}]{deps}")

### Run the Pipeline On-Demand

In [ ]:
# ============================================================================
# 11. Execute the pipeline
# ============================================================================

# Important: register the RAG tool on pipeline agents before running
# (The pipeline factory creates SingleAssistantRAG agents automatically
#  when retrieve_config is in task.metadata)

# ⚠️ The default factory creates agents with their own RAG tool.
# For a custom tool like query_local_rag, we need to post-register it.
# Here we use a custom factory that registers query_local_rag on each agent.

def _pipeline_agent_factory(task, llm_cfg):
    """Custom factory that builds agents with query_local_rag tool."""
    from fin_ai.agents.workflow import SingleAssistantRAG
    retrieve_cfg = task.metadata.get("retrieve_config", {})
    if retrieve_cfg:
        agent = SingleAssistantRAG(
            task.agent_config,
            llm_config=llm_cfg,
            retrieve_config=retrieve_cfg,
            human_input_mode="NEVER",
            max_consecutive_auto_reply=8,
        )
        # Register the local RAG tool on this agent
        register_function(
            query_local_rag,
            caller=agent.assistant,
            executor=agent.user_proxy,
            name="query_local_rag",
            description=query_local_rag.__doc__,
        )
        # Also register YFinance for Market_Analyst
        if task.agent_config in ("Market_Analyst",):
            register_function(
                yf_tools.get_stock_info,
                caller=agent.assistant,
                executor=agent.user_proxy,
                name="get_stock_info",
                description="Get latest stock information for a ticker.",
            )
        return agent
    else:
        from fin_ai.agents.workflow import SingleAssistant
        return SingleAssistant(
            task.agent_config,
            llm_config=llm_cfg,
            human_input_mode="NEVER",
        )

print("▶️  Running pipeline...")
result = pipeline.run(agent_factory=_pipeline_agent_factory)

print(f"\n{'='*60}")
print(f"Pipeline: {result.pipeline_name}")
print(f"Success: {result.success}")
print(f"Duration: {result.completed_at - result.started_at}")
if result.errors:
    print(f"Errors: {result.errors}")
print(f"\nTask outputs:")
for name, content in result.task_results.items():
    preview = str(content)[:300].replace('\n', ' ') if content else "(no output)"
    print(f"  [{name}]: {preview}...")

### Scheduled Execution (Background)

For recurring analysis, use ``AgentScheduler``:

In [ ]:
# ============================================================================
# 12. Schedule recurring pipeline runs (optional)
# ============================================================================

scheduler = AgentScheduler(
    pipelines=[pipeline],
    interval_seconds=3600,   # Run every hour
    run_on_start=False,      # Don't run immediately
)

# Uncomment to start background scheduling:
# scheduler.start()
# print("⏰ Scheduler running. Press Ctrl+C or call scheduler.stop() to stop.")

print("Scheduler ready. Uncomment scheduler.start() to activate background runs.")

---

## Direct Query (No Agent) — Using the Dashboard Engine

For quick, non-agentic queries, call ``answer_question`` directly:

In [ ]:
# ============================================================================
# 13. Direct query (fast path — no agent overhead)
# ============================================================================

if source_configs:
    direct_result = answer_question(
        question="What are the key financial highlights for NVIDIA?",
        source_configs=source_configs,
        provider=chat_provider,
        system_prompt="You are a concise financial analyst. Answer from the context.",
        temperature=0.1,
        retrieval_mode="ensemble",
    )
    print(f"⏱️  Answered in {direct_result['elapsed']:.1f}s\n")
    print(direct_result["response"][:1500])
else:
    print("No vector stores loaded. Skipping direct query.")

---

## Summary

| Approach | When to use | Key API |
|----------|-------------|---------|
| **Direct `answer_question`** | Quick lookup, no reasoning needed | `fin_ai.core.fin_ai_engine.answer_question()` |
| **SingleAssistantRAG** | Single agent with RAG + reasoning | `fin_ai.agents.workflow.SingleAssistantRAG` |
| **AgentPipeline** | Multi-agent workflows with dependencies | `fin_ai.agents.scheduler.AgentPipeline` |
| **AgentScheduler** | Recurring background analysis | `fin_ai.agents.scheduler.AgentScheduler` |
| **engine_bridge** | Agents → local RAG + model listing | `fin_ai.agents.engine_bridge` |

### Files used

| File | Role |
|------|------|
| `src/fin_ai/agents/workflow.py` | `FinRobot`, `SingleAssistant`, `SingleAssistantRAG`, `register_tools` |
| `src/fin_ai/agents/engine_bridge.py` | `query_local_rag`, `list_available_models`, `get_financial_snapshot`, `init_engine` |
| `src/fin_ai/agents/scheduler.py` | `AgentTask`, `AgentPipeline`, `AgentScheduler` |
| `src/fin_ai/agents/agentic_rag.py` | `get_rag_function` (autogen file-based RAG) |
| `src/fin_ai/agents/agent_library.py` | Agent profiles: `Data_Analyst`, `Market_Analyst`, `Research_Analyst`, `Thematic_Investor` |
| `src/fin_ai/core/fin_ai_engine.py` | `load_vector_stores_for_query`, `answer_question`, `fetch_models` |
| `src/fin_ai/core/tools.py` | YFinance tools + `YAHOO_FINANCE_TOOLS` schemas |
| `dashboard/financial_analyst_dashboard.py` | Streamlit UI (reference) |
| `dashboard/utils.py` | `get_embeddings`, PDF/CSV rendering |